# nano Beatrix — the longer run: the key-hard pointer recipe at 48,000 steps, and Alhena-P at the same length

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_keypointer_long.ipynb)

**The question**: Alhena answers a question exactly when it is handed the address it asks for, and dozens of independently trained arms then merge under the gate with nothing lost; the address has to come from the model itself. The key-hard pointer screen (`colab/nano_keypointer.ipynb`) trained the told head (block 4, head 0) on rows where only the whole key finds the record. On one seed of six the head reached the whole key after a sudden jump at 16,000-18,000 steps: its request and the index that model writes for itself were both right on 99.98% of the questions, and 48 arms merged through either lost nothing. On the other five the head still matched the key's last two characters and, on the hard rows, spread its attention evenly over the four records that share them; their training signal sat where that even split puts it (ln 4 / 2 = 0.693) and was still falling slowly at 24,000 steps.

**This notebook** trains the same recipes for twice as long (48,000 steps; the learning-rate schedule's cosine over the longer run, nothing else changed):

| part | what | why |
|---|---|---|
| PKL s1, s2, s3, s5, s6 | Alhena-PK at 48,000 steps (series `keypointerlong`) | do the seeds that stalled reach the whole key with more time |
| PL s1, s2 | Alhena-P at 48,000 steps (series `pointerlong`) | does the length alone, without the key-hard rows, take the head to the whole key |
| PK s7, s8 | Alhena-PK at 24,000 steps (series `keypointerrep`) | the replication arm |
| members | the scale test's 48 real arms + 48 toy members (after the last block) and 16 real arms (after block 3), each trained alone, on up to three 48,000-step models whose look on the model's own row is right on at least .98 of the questions | merged under w_i = sinh(u_i) / (1 + sum_j (cosh(u_j) - 1)), the look as the request |

**The cells**: C1 pins the trainer and the arm library and holds the decision block; C2 builds the data, runs 200 steps of the first part on this card for the price line (a projection past 1.5x the plan stops for a re-plan) and probes the push path; C3 trains the nine models one at a time and reads each one's look as it lands; C4 applies the member rule; C5 trains the members; C6 renders the tables. Every finished run and record ships to [AbstractPhil/alephllm-nano-beatrix-training](https://huggingface.co/AbstractPhil/alephllm-nano-beatrix-training) (`bed/reports/`, `bed/weights/<run>/`, `bed/reports/look/`, `bed/scale_members/`).

**Before C1**: the notebook's `HF_TOKEN` secret (write access to that repo) with notebook access on. Run C1, then Runtime > Restart session and run all. A file named `STOP` in `/content/out` ends the session between runs; a restarted session resumes where it stopped. The code and its tests: [AbstractEyes/geolip-nano-beatrix-trainer](https://github.com/AbstractEyes/geolip-nano-beatrix-trainer) (`geolip/nano_beatrix/bed_train.py`, `grid_rows.py`, `pointer.py`, `rotary_scale.py`).

In [ ]:
# C1 — pinned installs + THE DECISION BLOCK. RESTART the session after the first run of this cell (Runtime > Restart session and run all).
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # a progress bar is not allowed to cost a session
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.80" "amoe-lora @ git+https://github.com/AbstractEyes/amoe-lora@v0.2.12"
import torch, amoe
import geolip.nano_beatrix as N
print('geolip.nano_beatrix', N.__version__, '| amoe', amoe.__version__, '| torch', torch.__version__, '| card',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 80), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart session, then rerun from C1')
assert amoe.__version__ == '0.2.12', 'the arm module of record is amoe-lora 0.2.12'
assert torch.cuda.is_available(), 'no card: Runtime > Change runtime type > GPU'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN            # the runner and the shipper read it from the environment; the value is never printed

# ============ THE DECISION BLOCK — every setting set from the plan, with its source beside it ============
OUT = '/content/out'                                   # the session folder: the corpus, the records, the weights, the looks, the members
REPO = 'AbstractPhil/alephllm-nano-beatrix-training'  # every finished run and record ships here; tokenless = nothing ships
CORPUS_BYTES = 160e6                                   # fineweb-edu text for the bed (the scale test's corpus)
BED = dict(                                            # the scale test's Alhena recipe at 24,000 steps, verbatim
    ctx=4096, mb=8, steps=24000,                       # 24,000 steps x 8 rows x 4,096 bytes; pure Adam, lr 1e-3, 100 warm-up steps, the cosine to a tenth
    frames='dyn', plant=1.0,
    sizes='32,128,256,512', frame_records=4, nest=0.3, overlap=0.0, ask='any', register='caption', tags='mode',
    log_every=100,
)
LONG = dict(steps=48000)                               # THE LONGER RUN: 48,000 steps, the cosine over the run (the only change)
GRID_ROWS = dict(grid='ag', grid_p=0.5,                # half of every batch's rows are addressed grid rows
                 head='digit', head_lam=1.0,           # the per-digit address head at the questions' SEPs (the scale test's term)
                 trace_every=2000,                     # the formation trace every 2,000 steps (the pointer's masses read at each point)
                 hard_read=1)                          # the index on 64 key-hard rows at the end
WIDE = dict(rotary_content=4)                          # ALHENA-W: 4 position / 4 content / 8 address pairs
VALUE = dict(value_addr=1)                             # ALHENA-V: the value-path tables, born zero (the name carries _va1)
POINTER = dict(ptr_lam=0.1, ptr_block=4, ptr_head=0)  # ALHENA-P: block 4 of 6, head 0, lambda .1 (the pointer screen's rule)
HARD = dict(grid_hard=0.5)                             # ALHENA-PK: the last 2 of the batch's 4 grid rows key-hard, from birth (the name carries _kh50)
TAIL = lambda L: f'_L{L}_ctx4096_frdyn_cap_mode_sz32-128-256-512_r4_n30_o0_askany_gag50'
TAG_PKL = lambda s: f'alhena-both_s{s}{TAIL(48000)}_cp4_kh50_va1_pt10b4h0_ahdigit1_keypointerlong'
TAG_PL = lambda s: f'alhena-both_s{s}{TAIL(48000)}_cp4_va1_pt10b4h0_ahdigit1_pointerlong'
TAG_PKR = lambda s: f'alhena-both_s{s}{TAIL(24000)}_cp4_kh50_va1_pt10b4h0_ahdigit1_keypointerrep'
PKL = lambda s: (f'PKL s{s}', TAG_PKL(s), s, dict(BED, **LONG, **GRID_ROWS, **HARD, **WIDE, **VALUE, **POINTER, arms='alhena-both', seeds=str(s), series='keypointerlong'))
PL = lambda s: (f'PL s{s}', TAG_PL(s), s, dict(BED, **LONG, **GRID_ROWS, **WIDE, **VALUE, **POINTER, arms='alhena-both', seeds=str(s), series='pointerlong'))
PKR = lambda s: (f'PK s{s}', TAG_PKR(s), s, dict(BED, **GRID_ROWS, **HARD, **WIDE, **VALUE, **POINTER, arms='alhena-both', seeds=str(s), series='keypointerrep'))
PARTS = [PKL(1), PL(1), PKL(2), PKR(7), PKL(3), PL(2), PKL(5), PKR(8), PKL(6)]   # the plan's order: the 24,000-step replication spread through
MEMBER_ARMS = ('PKL', 'PL')                            # the member rule's arms: the 48,000-step cores only, never the replication arm
LOOK_HEAD = (4, 0)                                     # block 4 (numbered from 1), head 0: the head the term trains
MIN_LOOK, MIN_ANSWERS = 0.98, 0.95                     # the member rule: the look (free-running) right >= .98, the answers with the true address >= .95
N_MEMBER_CORES = 3                                     # at most three member cores, the look's right highest first
LAST_STEPS, MID_STEPS = 16000, 32000                   # the scale test's member budget, verbatim
READ_BATCHES, CALIB_BATCHES, HARD_BATCHES = 64, 32, 16 # 512 read rows, 256 calibration rows, 128 key-hard rows (8 rows a batch)
SMOKE_STEPS = 200                                      # C2's short run of the first part on this card for the price line; its record stays local
PRICE_STEPS = 200                                      # the member price lines (each trainer on throwaway stacks)
PLAN_MIN_24K = 17.4                                    # the planned minutes of one 24,000-step core on this card (PK measured 17.16-17.22); a part's plan scales with its steps
PLAN_MIN_MEM = 18.8                                    # the planned training minutes of one core's members (4.9 last + 13.8 mid measured)
STOP_FACTOR = 1.5                                      # a projection past 1.5x the plan stops the session for a re-plan
PARALLEL = 1                                           # ONE run at a time: one 9M craft at 8 x 4,096 saturates the card
print('decision block:', [p[0] for p in PARTS], '| steps', [p[3]['steps'] for p in PARTS], '| the member rule', MIN_LOOK, MIN_ANSWERS,
      'at most', N_MEMBER_CORES, 'of', MEMBER_ARMS, '| out', OUT, '| ships to', REPO if HF_TOKEN else 'nothing (tokenless)')

In [ ]:
# C2 (P) — PREFLIGHT: the data, the plan, 200 steps of the first part on this card (the price line), the push probe. No part trains.
import json, os, shutil, time, torch
from geolip.nano_beatrix import bed_corpus, bed_runner
from geolip.nano_beatrix import rotary_scale as rs
if not os.path.exists(f'{OUT}/bed/bed_corpus.bin'):
    bed_corpus.build(f'{OUT}/bed', CORPUS_BYTES)
HELD = f'{OUT}/bed/heldout_fineweb_edu_head2048.jsonl'
print('corpus', os.path.getsize(f'{OUT}/bed/bed_corpus.bin') / 1e6, 'MB; held-out head', os.path.exists(HELD))
todo_steps = 0
for label, tag, seed, spec in PARTS:
    p = bed_runner.run(spec, out=OUT, dry=True)
    todo_steps += (p['runs'] - p['done_before']) * spec['steps']
smoke = dict(PARTS[0][3], steps=SMOKE_STEPS, log_every=20, trace_every=100, read_batches=4, calib_batches=2, hard_batches=1)
res = bed_runner.run(smoke, out=OUT)
assert res['failed'] is None, 'the short run failed: read the lines above before any part'
last = [s for s in (bed_runner.parse_step(l) for l in open(f'{OUT}/runner.log', encoding='utf-8')) if s][-1]
per_24k = 24000 * BED['ctx'] * BED['mb'] / (last['tok_s'] * 1e3) / 60
PRICE_OK = per_24k <= STOP_FACTOR * PLAN_MIN_24K
print(f"this card: {last['tok_s']}k tok/s at {BED['ctx']} x {BED['mb']} -> one 24,000-step run about {per_24k:.1f} min, a 48,000-step run "
      f"about {2 * per_24k:.1f} min (planned {PLAN_MIN_24K} and {2 * PLAN_MIN_24K}); the {todo_steps:,} steps left about "
      f"{per_24k * todo_steps / 24000 / 60:.1f} h (+ a few minutes of end reads and the look read a run, + the members)"
      + ('' if PRICE_OK else f' -> PAST {STOP_FACTOR}x THE PLAN: C3 will not start; re-plan first'))
PUSH_OK = False
if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    path = 'bed/preflight/push_probe_keypointer_long.json'
    body = json.dumps({'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'card': torch.cuda.get_device_name(0),
                       'tok_s': last['tok_s'], 'parts': [p[0] for p in PARTS]}).encode()
    api.upload_file(path_or_fileobj=body, path_in_repo=path, repo_id=REPO, commit_message='nano keypointer long: push probe')
    PUSH_OK = path in set(api.list_repo_files(REPO))
    print('push probe', 'READ BACK OK' if PUSH_OK else 'NOT VISIBLE after upload', path)
else:
    print('TOKENLESS: no probe written; the cells below run and ship nothing')

def ship_file(path, dst_dir, ship):
    # one record to dst_dir on the Hub: staged alone in its own folder, so the upload carries only it
    st = f'{OUT}/stage/{os.path.basename(path)[:-5]}'
    os.makedirs(st, exist_ok=True)
    shutil.copy(path, st)
    ship.submit(st, dst_dir)

In [ ]:
# C3 (T) — THE NINE MODELS, one run at a time in the plan's order, resume-first (runner_done.txt in OUT; a look record in OUT/look/ is not
# read again); a STOP file in OUT starts no new run; every finished run ships to REPO as it lands, and its look read (the told head's
# request on 512 read rows and 128 key-hard rows) right after it, shipped to bed/reports/look/ (background shippers: the card never
# waits on the Hub). The time left is priced by the steps left at this session's measured pace.
assert PRICE_OK, 'the price line projected past the plan: re-plan before any part trains'
push = REPO if (HF_TOKEN and PUSH_OK) else ''
shipper = globals().get('shipper') or (bed_runner._Shipper(push, OUT, print) if push else None)
if shipper:
    for rec in bed_runner.pending_records(OUT):
        shipper.submit(rec)
lship = rs.Shipper(REPO) if push else None
done, t_all, steps_done = [], time.time(), 0
for n, (label, tag, seed, spec) in enumerate(PARTS, 1):
    if os.path.exists(f'{OUT}/STOP'):
        print('STOP present: no new part starts')
        break
    print(f"=================== {label} ({n}/{len(PARTS)}, {spec['steps']:,} steps) ===================")
    t0 = time.time()
    res = bed_runner.run(spec, out=OUT, push=push, parallel=PARALLEL, shipper=shipper)
    done.append((label, res))
    if res['failed'] is not None or res['stopped']:
        print('the part did not finish:', label, {k: v for k, v in res.items() if k != 'ran'})
        break
    assert os.path.exists(f'{OUT}/bed/{tag}/model.safetensors'), ('the run did not leave the expected folder', tag)
    out = f'{OUT}/look/{rs.look_name(tag, LOOK_HEAD)}.json'
    rec = rs.look_read(f'{OUT}/bed/{tag}', out, seed, HELD, look='ptr', read_batches=READ_BATCHES, hard_batches=HARD_BATCHES,
                       log=lambda s: None)
    assert (rec['look']['block'], rec['look']['head']) == LOOK_HEAD, (label, rec['look'])
    if lship:
        ship_file(out, 'bed/reports/look', lship)
    q, h = rec['read']['request'], rec['hard']['request']
    steps_done += spec['steps'] if res['ran'] else 0
    left = sum(p[3]['steps'] for p in PARTS[n:])
    pace = (time.time() - t_all) / max(1, steps_done)
    print(f"{label}: the look on the model's own row {q['LOOKF']['right']:.4f} (composed row {q['LOOK']['right']:.4f}; on the key-hard rows "
          f"{h['LOOKF']['right']:.4f}); the written index {q['INDEXF']['right']:.4f}; {(time.time() - t0) / 60:.1f} min; "
          f"{(time.time() - t_all) / 60:.1f} min spent; {len(PARTS) - n} models left ({left:,} steps), about {pace * left / 3600:.1f} h")
if shipper:
    shipper.drain(print)
    left = [os.path.basename(p)[4:-5] for p in bed_runner.pending_records(OUT)]
    print('UNSHIPPED runs:', left if left else 'none')
if lship:
    print('look records shipped' if lship.drain() else 'look records still retrying in the background')
print('session:', [(n, r['failed'], r['stopped'], len(r['ran'])) for n, r in done])

In [ ]:
# C4 — THE MEMBER RULE (the plan's, written before any run): the 48,000-step models (PKL, PL) whose look on their own row is right on
# >= .98 of the read's questions and whose answers with the true address reach .95, the look's right highest first, at most three; never
# a replication core.
cand = []
for label, tag, seed, spec in PARTS:
    if label.split()[0] not in MEMBER_ARMS:
        continue
    p = f'{OUT}/look/{rs.look_name(tag, LOOK_HEAD)}.json'
    if not os.path.exists(p):
        print(f'{label}: no look record in this session')
        continue
    rec = json.load(open(p, encoding='utf-8'))
    lf, ans = rec['read']['request']['LOOKF']['right'], rec['read']['answers']['LAYOUT']
    ok = lf >= MIN_LOOK and ans >= MIN_ANSWERS
    print(f'{label}: the look (free-running) {lf:.4f}, the answers with the true address {ans:.4f} -> ' + ('a member core' if ok else 'not'))
    if ok:
        cand.append((lf, label, tag, seed))
cand.sort(key=lambda c: -c[0])                         # the look's right, highest first (ties keep the plan's order)
MEMBER_CORES = [(label, tag, seed) for _, label, tag, seed in cand[:N_MEMBER_CORES]]
print('member cores, in order:', [c[0] for c in MEMBER_CORES] or 'none')

In [ ]:
# C5 (T) — THE MEMBERS on C4's member cores, in its order: the scale test's recipe verbatim (48 real arms + 48 toy members after block 6,
# 16 real arms after block 3, each trained alone on the frozen model), read in the two-pass form with the look sources; one core at a time,
# resume-first; each core's members ship to bed/scale_members/ as they land.
def run_members(cores):
    owned, never = rs.owned_sets()
    packed = rs.load_packed(f'{OUT}/bed')
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    core = rs.Core(f'{OUT}/bed/{cores[0][1]}', 'cuda')
    cc = rs.compile_check(core, packed, owned, never)
    print('compiled = eager at step 1 (largest difference over the largest value):', cc)
    assert all(v['write'] < 1e-4 and v['grad'] < 1e-3 for v in cc.values()), 'the compiled writes are not the eager writes: no member trains'
    pr = rs.price(core, packed, owned, never, steps=PRICE_STEPS)
    per_core = (pr['last_ms_per_step'] * LAST_STEPS + pr['mid_ms_per_step'] * MID_STEPS) / 60000
    print(f"this card: last site {pr['last_ms_per_step']:.1f} ms a step, mid site {pr['mid_ms_per_step']:.1f} ms a step -> a core's members "
          f"about {per_core:.0f} min of training (planned {PLAN_MIN_MEM}) + the reads")
    assert per_core <= STOP_FACTOR * PLAN_MIN_MEM, 'the members priced past the plan: re-plan before any member trains'
    del core
    torch.cuda.empty_cache()
    ship = rs.Shipper(REPO) if (HF_TOKEN and PUSH_OK) else None
    t_all = time.time()
    out = []
    for n, (label, tag, seed) in enumerate(cores, 1):
        if os.path.exists(f'{OUT}/STOP'):
            print('STOP present: no new core starts')
            break
        print(f'=================== members {n}/{len(cores)} {label} ===================')
        t0 = time.time()
        res = rs.run_core(f'{OUT}/bed/{tag}', f'{OUT}/members', seed, f'{OUT}/bed', HELD, last_steps=LAST_STEPS, mid_steps=MID_STEPS,
                          read_batches=READ_BATCHES, calib_batches=CALIB_BATCHES, shipper=ship, forms=('find',))
        out.append(res)
        torch.cuda.empty_cache()
        print(f'{label}: {(time.time() - t0) / 60:.1f} min; {(time.time() - t_all) / 60:.1f} min spent; {len(cores) - n} cores left')
    if ship:
        print('ships drained' if ship.drain() else 'ships still retrying in the background')
    return out

done_m = run_members(MEMBER_CORES) if MEMBER_CORES else []
print('members:', [r['name'] for r in done_m] or 'none (no 48,000-step model passed the rule)')

In [ ]:
# C6 — THE TABLES: the models (bits, the frame copy, the written index both ways, the key-hard rows both ways, the answers, the told head's
# mass at steps 2,000 / 4,000 / the end read, minutes), their looks (the requests' right, the look's no-request and wrong-record shares,
# the answers through each, the key-hard rows), and the members (per core and stack: solo against bare, the nine rest cells keeping all
# three marks and QUIET per source, the controls, the largest leak through the look, the mid site, the text bits). Run any time.
import glob, json
from IPython.display import Markdown, display
rows = ['| run | bits | copy 32 / 128 / 256 / 512 | index TF / FR | key-hard rows TF / FR | answers true / via index | pointer mass 2k / 4k / read | min |',
        '|---|---|---|---|---|---|---|---|']
for label, tag, seed, spec in PARTS:
    p = f'{OUT}/bed/bed_{tag}.json'
    if not os.path.exists(p):
        continue
    r = json.load(open(p, encoding='utf-8'))
    ff = r.get('fetch_by_frame') or {}
    copy = ' / '.join(f"{ff[f'data/{s}/depth0']['full_value']:.3f}" if f'data/{s}/depth0' in ff else '-' for s in (32, 128, 256, 512))
    g = r['grid']['read']
    fr = g.get('free_running') or {}
    hd = g.get('hard')
    hard = f"{hd['index_right']:.4f} / {hd['indexf_right']:.4f}" if hd else '-'
    via = f"{g['true']['whole']:.4f} / {fr.get('indexf_whole', float('nan')):.4f}"
    pt_ = r['grid'].get('ptr') or {}
    tm = {t['step']: t.get('ptr_mass') for t in r['grid']['trace']}
    f_ = lambda v: '-' if v is None else f'{v:.3f}'
    ptr = f"{f_(tm.get(2000))} / {f_(tm.get(4000))} / {f_(pt_.get('mass_read'))}"
    rows.append(f"| {label} | {r['heldout_bpb']:.4f} | {copy} | {g['true']['index_right']:.4f} / {fr.get('indexf_right', float('nan')):.4f} | "
                f"{hard} | {via} | {ptr} | {r['train_minutes']:.1f} |")
display(Markdown('\n'.join(rows)))
rows = ['| model | written index (free-running) | look: composed / free-running / at the question | free-running look: none / wrong record | '
        'answers: true / written / look | look mass (decoded) / split / duplicate | key-hard rows: written / look | min |',
        '|---|---|---|---|---|---|---|---|']
for label, tag, seed, spec in PARTS:
    p = f'{OUT}/look/{rs.look_name(tag, LOOK_HEAD)}.json'
    if not os.path.exists(p):
        continue
    r = json.load(open(p, encoding='utf-8'))
    q, a, lk = r['read']['request'], r['read']['answers'], r['read']['look']['LOOKF']
    h = r.get('hard', {}).get('request', {})
    hard = f"{h['INDEXF']['right']:.4f} / {h['LOOKF']['right']:.4f}" if h else '-'
    rows.append(f"| {label} | {q['INDEXF']['right']:.4f} | {q['LOOK']['right']:.4f} / {q['LOOKF']['right']:.4f} / {q['LOOKQF']['right']:.4f} | "
                f"{lk['no_request']:.4f} / {lk['wrong_record']:.4f} | {a['LAYOUT']:.4f} / {a['INDEXF']:.4f} / {a['LOOKF']:.4f} | "
                f"{lk['decoded_mass_mean']:.3f} / {lk['split']:.4f} / {lk['duplicate']:.4f} | {hard} | {r['wall_s'] / 60:.1f} |")
display(Markdown('\n'.join(rows)))
allm = lambda x: all(x[k] for k in ('PROTECTED', 'QUIET', 'EXACT'))
def _nine(c, s, mark=None):
    if f'{s}_U5_REST_m16' not in c:
        return '-'
    ks = [f'{s}_U{u}_REST_m{m}' for u in (5, 8, 10) for m in (16, 32, 48)]
    return f"{sum(allm(c[k]) if mark is None else c[k][mark] for k in ks)}/9"
rows = ['| core | stack | solo / bare owned | look free-running: all three / QUIET | written: all three / QUIET | true address | look composed | '
        'look at the question | G2 m48 | INDEXC m48 Q | G0 m48 P/Q | largest leak / own (look) | mid look U5/8/10 | G0 text bits |',
        '|---|---|---|---|---|---|---|---|---|---|---|---|---|---|']
for d in sorted(glob.glob(f'{OUT}/members/*_members_*')):
    lp, mp = f'{d}/last_find.json', f'{d}/mid_find.json'
    if not os.path.exists(lp):
        continue
    L = json.load(open(lp, encoding='utf-8'))
    rd = L['read']
    M = json.load(open(mp, encoding='utf-8')) if os.path.exists(mp) else None
    for st, r in rd['stacks'].items():
        c = r['cells']
        leak = (max(c[f'LOOKF_U{u}_REST_m{m}']['leak_over_own_write'] for u in (5, 8, 10) for m in (16, 32, 48))
                if 'LOOKF_U5_REST_m16' in c else float('nan'))
        mid = '-'
        if M and st == 'arms':
            mc = M['read']['cells']
            mid = ' '.join(str(allm(mc[f'LOOKF_U{u}_REST_m16'])) if f'LOOKF_U{u}_REST_m16' in mc else '-' for u in (5, 8, 10))
        g0 = (L.get('text_bits') or {}).get(f'G0_{st}_minus_bare')
        rows.append(f"| {L['core'][:30]} | {st} | {r['solo']['ONE']['48']:.3f} / {rd['bare']['owned']:.3f} | "
                    f"{_nine(c, 'LOOKF')} / {_nine(c, 'LOOKF', 'QUIET')} | {_nine(c, 'INDEXF')} / {_nine(c, 'INDEXF', 'QUIET')} | "
                    f"{_nine(c, 'LAYOUT')} | {_nine(c, 'LOOK')} | {_nine(c, 'LOOKQF')} | {allm(c['G2_m48'])} | {c['INDEXC_U5_REST_m48']['QUIET']} | "
                    f"{c['G0_m48']['PROTECTED']}/{c['G0_m48']['QUIET']} | {leak:.1e} | {mid} | " + ('-' if g0 is None else f'{g0:+.3f}') + ' |')
display(Markdown('\n'.join(rows)))